# Song Lyrics Generator Using Subword Tokenization and LSTM in PyTorch

- Author: Jovan Heng Ghim Hong 

Goal: Build a model on pop song lyrics to generate lyrics in this style. Explore the use of subword tokenization and LSTM for text generation.

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
#imports
import os
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# import hugging face tokenizer
from transformers import BertTokenizer

# torch
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# setting seeds
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

from sklearn.model_selection import train_test_split

from src.model import Model
from src.utils import timer, DeviceLoader

# setting device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [3]:
# data processing has been done inside etl.py
df = pd.read_parquet("data/pop_songs.parquet")

In [4]:
display(df.head())

,tag,title,lyrics
0,pop,Wordy Rappinghood,<CHORUS> <NEWLINE> What are words worth? <NE...
1,pop,Horchata,"<VERSE> <NEWLINE> In December, drinking horc..."
2,pop,Heartless,"<CHORUS> <NEWLINE> In the night, I hear 'em ..."
3,pop,Flashing Lights,"<INTRO> <NEWLINE> Flashing lights (Lights, l..."
4,pop,Baby,"<NEWLINE> <INTRO> <NEWLINE> Oh, woah <NEWLI..."


In [5]:
# train test validation split 60-20-20
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.25, random_state=42)

In [6]:
# create tokenizers for training, validation and testing
tokenizer_train = BertTokenizer.from_pretrained("bert-base-uncased")
tokenizer_val = BertTokenizer.from_pretrained("bert-base-uncased")
tokenizer_test = BertTokenizer.from_pretrained("bert-base-uncased")

# add the <VERSE> tokens to the tokenizers
sections = ['VERSE', 'CHORUS', 'BRIDGE', 'INTRO', 'OUTRO', 'PRE-CHORUS', 'HOOK']
tokenizer_train.add_tokens([f"<{section}>" for section in sections])
tokenizer_val.add_tokens([f"<{section}>" for section in sections])
tokenizer_test.add_tokens([f"<{section}>" for section in sections])

# tokenizer the lyrics
train_X = tokenizer_train(train_df["lyrics"].tolist(), padding=True, truncation=True, return_tensors="pt")["input_ids"]
val_X = tokenizer_val(val_df["lyrics"].tolist(), padding=True, truncation=True, return_tensors="pt")["input_ids"]
test_X = tokenizer_test(test_df["lyrics"].tolist(), padding=True, truncation=True, return_tensors="pt")["input_ids"]

# create the Y labels by shifting
train_Y = torch.zeros_like(train_X)
train_Y[:, :-1] = train_X[:, 1:]
train_Y[:, -1] = tokenizer_train.pad_token_id

val_Y = torch.zeros_like(val_X)
val_Y[:, :-1] = val_X[:, 1:]
val_Y[:, -1] = tokenizer_val.pad_token_id

test_Y = torch.zeros_like(test_X)
test_Y[:, :-1] = test_X[:, 1:]
test_Y[:, -1] = tokenizer_test.pad_token_id

In [15]:
import os
# Save tokenized tensors for reuse
output_dir = "data/tokenized"
os.makedirs(output_dir, exist_ok=True)

vocab_size = len(tokenizer_train)

torch.save(
  {
    "train_X": train_X,
    "train_Y": train_Y,
    "val_X": val_X,
    "val_Y": val_Y,
    "test_X": test_X,
    "test_Y": test_Y,
    "vocab_size": vocab_size,
    "pad_token_id": tokenizer_train.pad_token_id,
    "sections": sections,
  },
  f"{output_dir}/tokenized_lyrics.pt",
)

print(f"Saved tokenized outputs to {output_dir}/tokenized_lyrics.pt")

Saved tokenized outputs to data/tokenized/tokenized_lyrics.pt


In [12]:
# save the tokenizer artifacts for reuse
tokenizer_dir = "data/tokenizer"
os.makedirs(tokenizer_dir, exist_ok=True)
tokenizer_train.save_pretrained(tokenizer_dir)
tokenizer_val.save_pretrained(tokenizer_dir)
tokenizer_test.save_pretrained(tokenizer_dir)

('data/tokenizer\\tokenizer_config.json',
 'data/tokenizer\\special_tokens_map.json',
 'data/tokenizer\\vocab.txt',
 'data/tokenizer\\added_tokens.json')

In [13]:
# let us create a basic lstm model
class LSTMModel(Model):
  def __init__(self, vocab_size, input_dim, hidden_dim, output_dim, dropout_rate=0.5):
    super().__init__()
    self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=input_dim, padding_idx=0)
    self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True, bidirectional=False)
    self.dropout = nn.Dropout(dropout_rate)
    self.fc = nn.Linear(hidden_dim, output_dim)
  
  def forward(self, x):
    x = self.embedding(x)
    h, _ = self.lstm(x)
    h = self.dropout(h)
    out = self.fc(h)
    return out

In [16]:
# load the tokenized tensors
tokenized_data = torch.load("data/tokenized/tokenized_lyrics.pt")
train_X = tokenized_data["train_X"]
train_Y = tokenized_data["train_Y"]
val_X = tokenized_data["val_X"]
val_Y = tokenized_data["val_Y"]
test_X = tokenized_data["test_X"]
test_Y = tokenized_data["test_Y"]
vocab_size = tokenized_data["vocab_size"]
pad_token_id = tokenized_data["pad_token_id"]

In [18]:
# build dataloaders
batch_size = 16

train_dataset = TensorDataset(train_X, train_Y)
val_dataset = TensorDataset(val_X, val_Y)
test_dataset = TensorDataset(test_X, test_Y)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

train_loader_device = DeviceLoader(train_loader, device)
val_loader_device = DeviceLoader(val_loader, device)

In [20]:
# model setup
model = LSTMModel(
  vocab_size=vocab_size,
  input_dim=128,
  hidden_dim=256,
  output_dim=vocab_size,
  dropout_rate=0.3
).to(device)

loss_fn = nn.CrossEntropyLoss(ignore_index=pad_token_id)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
# train
epochs = 30
with timer("Baseline LSTM: "):
  model.fit(
    epochs=epochs,
    train_loader=train_loader_device,
    loss_fn=loss_fn,
    optimizer=optimizer,
    val_loader=val_loader_device,
    early_stop=True,
    patience=3
  )


Baseline LSTM: : 929.4389s


KeyboardInterrupt: 

In [ ]:
# save model weights
model_dir = "models/lstm"
os.makedirs(model_dir, exist_ok=True)
torch.save(model.state_dict(), f"{model_dir}/lstm_model.pt")

In [ ]:

plt.figure(figsize=(10, 4))
plt.plot(model.loss_history, label="Train Loss")
plt.plot(model.val_loss_history, label="Val Loss")
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(model.perplexity_history, label="Train Perplexity")
plt.plot(model.val_perplexity_history, label="Val Perplexity")
plt.title("Training vs Validation Perplexity")
plt.xlabel("Epoch")
plt.ylabel("Perplexity")
plt.legend()
plt.show()